# MSE 3302B: Sensors and Actuators
## Actuation-Based Mechatronics for Rock-Paper-Scissors-Minus-One

**Western University, Faculty of Engineering — Department of Electrical and Computer Engineering**  
**Team MSE-08:** Evan Romano, Joseph Toma, Mohammed Alamen Qassab  
**Instructor:** Prof. Elvis Chen, Ph.D., LEL  
**Date:** March 2026  
**Video Demo:** [https://youtu.be/TVBDYf_IdJo](https://youtu.be/TVBDYf_IdJo)

## 1. Introduction


### 1.1 Background and Project Context

Rock-Paper-Scissors-Minus-One (RPS-1) is a two-stage variant of the classic hand game where each player simultaneously shows two hands, then withdraws one before the winner is determined by the remaining hands under standard RPS rules. The extra stage transforms what is normally a memoryless game into something with genuine strategic depth, because each player's Stage 2 decision is conditioned on observing what the opponent revealed at Stage 1.

As a mechatronics project, RPS-1 is compelling because it couples three distinct engineering subsystems under tight time constraints: a mechanical hand capable of forming three recognizable gestures on demand, a control system that actuates those gestures reliably and processes opponent input, and a decision engine that determines both what to show and what to keep each round. These subsystems cannot be designed in isolation. A strategy that picks a gesture the hand cannot form quickly enough is useless, and a hand that is mechanically capable but too heavy for its servos is equally useless. Getting all three to work together within 20 seconds for Stage 1 and 5 seconds for Stage 2 was the central engineering challenge.

### 1.2 Problem Statement and Objectives

The system had to meet the following functional requirements: produce mechanically stable Rock, Paper, and Scissors gestures with greater than 95% repeatability, complete all gesture transitions within the Stage 1 and Stage 2 time windows, implement a provably optimal game strategy using game-theoretic principles, and minimize component count and wiring complexity to reduce assembly burden and failure modes.

**In Scope:** Mechanical design and 3D printing of two articulated hands, closed-form analysis of servo performance and string-pulley mechanics, selection and integration of continuous rotation servos, open-loop timed control, serial-based opponent input, and game strategy using Nash equilibrium with an adaptive opponent layer.

**Out of Scope:** Computer vision gesture recognition, closed-loop encoder feedback, real-time opponent video processing, and advanced machine learning. These were excluded because they would have required extensive training data and additional hardware beyond the project timeline.

### 1.3 Report Organization

This report is organized as follows. Section 2 reviews candidate opponent input and actuation approaches from the literature and motivates the selected design. Section 3 describes the final system architecture, mechanical design, electrical and firmware implementation, and validation procedure. Section 4 presents calibration, timing, and gesture performance results. Section 5 discusses interpretation, uncertainty, and design trade-offs. Section 6 summarizes key findings and future improvements.

## 2. Literature Review

### 2.1 Opponent Sensing and Input Methods

To enable gameplay, the system needs opponent move input within a 5-second window. Three methods were considered.

Computer vision using a Raspberry Pi camera with a trained neural network could detect gestures autonomously, but it introduces model training complexity, requires controlled lighting and fixed camera geometry, and demands substantial additional compute and setup that were outside the project scope. The literature shows this is a mature domain, but a non-trivial one to deploy reliably in variable conditions.

A hardware button panel with labeled keys for each gesture combination is simple and reliable but adds mechanical complexity and still requires operator attentiveness under time pressure.

Serial monitor text input was selected. The operator types the opponent's two-hand combination into the Arduino IDE serial monitor within the 5-second window. This approach is simple to implement within the existing Arduino debugging workflow, provides clear echoed feedback to the operator and judges, and has no additional hardware dependencies. While it requires operator responsiveness, in practice it proved reliable and fast under tournament conditions. It also made the firmware logic straightforward to debug during development, since every input and output was visible in the serial monitor.

### 2.2 Actuation Technologies

Four motor types were evaluated for driving the string-pulley finger actuation system.

**Stepper motors** provide open-loop positional control without encoder feedback, which is attractive for a system with discrete gesture states. They were rejected because of their mass relative to the available palm space, their requirement for a dedicated stepper driver circuit not available on the Arduino Uno, and their continuous current draw at holding torque which complicates power budgeting on a single USB-powered board.

**N20 micro DC motors** are compact and high-speed, exceeding 15,000 RPM, which is favorable for space-constrained designs. They were evaluated for the Stage 2 withdrawal task in particular. The concern was that reliable closed-loop position control at the torques required for consistent finger actuation would require motor driver hardware adding both cost and firmware complexity. Without closed-loop control, open-loop duration timing on a small motor is sensitive to supply voltage variation. These concerns, combined with the absence of a motor driver in the available hardware kit, led to the N20 being set aside.

**Standard positional servos** are compact and support direct position commands over PWM, but they require knowing the target angle precisely. In a tendon system where string slack and elastic pre-tension shift the effective load continuously through the stroke, the relationship between commanded angle and actual finger position is not reliably linear. Position-mode servos tend to fight the mechanical end stops rather than settle gracefully, which causes string breakage over repeated cycles.

**FS90R continuous rotation servos** were selected. They interpret PWM as speed rather than position, which means the end state of a gesture is defined by the mechanical geometry of the finger and the string anchor point, not by a servo angle. This makes them well-suited for tendon systems with defined physical end stops. They are compact, inexpensive, and natively supported by the Arduino `Servo.h` library, requiring no additional driver hardware.

### 2.3 Candidate Comparison, Selection Criteria, and Research Gap

From the literature and our constraints, the deciding factors were repeatability, actuation speed, manufacturing complexity, integration effort on Arduino Uno, and robustness under limited build time.

For opponent move acquisition, we selected serial input over vision because it avoids dataset collection, model training, and camera calibration while still meeting the Stage 2 timing requirement. For actuation, we selected FS90R-based tendon drive because it keeps hardware overhead low and allows consistent end states through mechanical stops.

The gap we identified is practical guidance for low-cost, open-loop tendon hands that must work reliably under strict timing constraints in a student project setting. Many references focus on high-end dexterous systems with richer sensing, or on isolated mechanical concepts without an integrated calibration workflow. This project addresses that gap by combining tendon actuation, per-servo empirical tuning, and game-state control in one reproducible build.

## 3. Methods

### 3.1 System Overview

The RPS-1 actuation system consists of two 3D-printed articulated hands driven by continuous rotation servos using a string-pulley tendon mechanism with elastic band return. Opponent moves are entered manually through the Arduino serial monitor. The Arduino Uno processes the game strategy, selects gestures, and commands the servos to actuate the hands within the Stage 1 and Stage 2 time limits. This design prioritizes reliable mechanical actuation over autonomous sensing, using open-loop timed control calibrated empirically for repeatability.

### 3.2 Actuation Mechanism Selection

| Criterion | Weight | Rationale |
|-----------|--------|-----------|
| Precision / Repeatability | 40% | Inconsistent gesture formation is an immediate failure mode |
| Actuation Speed | 35% | Stage 1 limit 20 s, Stage 2 limit 5 s |
| Manufacturing Complexity | 20% | Two identical hands must be built within project timeline |
| Cost | 5% | Budget constraint |

| Criterion | Weight | String-Pulley | Direct Drive | 4-Bar Linkage |
|-----------|--------|--------------|--------------|---------------|
| Precision / Repeatability | 0.40 | +1 | 0 | -1 |
| Actuation Speed | 0.35 | 0 | +1 | +1 |
| Manufacturing Complexity | 0.20 | +1 | -1 | 0 |
| Cost | 0.05 | +1 | +1 | +1 |
| **Weighted Total** | 1.00 | **+0.65** | **+0.20** | **0.00** |

The string-pulley system was selected. It scores highest on precision, the most heavily weighted criterion, because the end position of each finger is defined by a physical anchor point rather than a servo angle, making it independent of supply voltage and per-unit servo variation. Direct drive was rejected for its manufacturing complexity: a servo per finger across two five-finger hands would require ten independently controlled channels. The 4-bar linkage was eliminated because the rigid geometry makes it difficult to reliably produce all three gestures without interference between link configurations.


### 3.3 Return Mechanism Selection

| Criterion | Weight | Rationale |
|-----------|--------|-----------|
| Motor Count | 40% | Fewer motors means lower cost, simpler wiring, simpler control |
| Manufacturability | 35% | Must produce two identical assemblies within project timeline |
| Reliability | 25% | Failure during gameplay results in forfeit |

| Criterion | Weight | Elastic Bands | Compression Springs | Opposing Motor |
|-----------|--------|--------------|---------------------|----------------|
| Motor Count | 0.40 | +1 | +1 | -1 |
| Manufacturability | 0.35 | +1 | -1 | -1 |
| Reliability | 0.25 | 0 | +1 | +1 |
| **Weighted Total** | 1.00 | **+0.75** | **+0.30** | **-0.50** |

Elastic bands were selected. They require no additional motors, can be installed and adjusted without precision machining, and provide sufficient return force for the finger mass at 5% infill PLA. Compression springs were rejected because precision spring cavities inside finger joints are difficult to print at the tolerances available on a hobby FDM machine, and cavity geometry that is off by even 0.5 mm produces inconsistent spring seating. The opposing motor option was eliminated by its penalties in both top criteria.


### 3.4 Finger Grouping Strategy

| Gesture | Thumb | Index | Middle | Ring | Pinky |
|---------|-------|-------|--------|------|-------|
| Rock | Curled | Curled | Curled | Curled | Curled |
| Paper | Extended | Extended | Extended | Extended | Extended |
| Scissors | Curled | Extended | Extended | Curled | Curled |

Index and Middle always move together across all three gestures. Thumb, Ring, and Pinky also always move together. This allows the ten finger-control channels across two hands to collapse to four, a 60% reduction in motor count. Two motors per hand is the theoretical minimum to distinguish all three gestures: a single motor cannot independently produce the Scissors configuration, where two fingers extend while three curl.

| Motor | Fingers | Function |
|-------|---------|----------|
| MI (Middle/Index) | Index + Middle | Controls scissors: extends these two fingers |
| PRT (Pinky/Ring/Thumb) | Thumb + Ring + Pinky | Controls the remaining group |

| Gesture | MI | PRT | Result |
|---------|-----|-----|--------|
| Rock | Curled | Curled | All fingers curled |
| Paper | Extended | Extended | All fingers extended |
| Scissors | Extended | Curled | Index/Middle extended, rest curled |


### 3.5 Mechanical Design and Fabrication

We printed both hands in PLA on a Creality Ender 3 at 5% gyroid infill. The gyroid pattern at this density produces an isotropic internal structure that is lightweight without sacrificing bending stiffness in any particular direction, which mattered for finger segments loaded from multiple angles depending on the gesture being held.

Finger segments were oriented vertically on the print bed so that layer lines run along the finger's length rather than across it. A finger segment printed flat would have layer lines perpendicular to the primary bending direction, the weakest orientation for FDM parts under cyclic load. Printing upright eliminates that failure mode. The curved knuckle geometry was printed without supports by relying on PLA's natural overhang tolerance, keeping the joint surfaces clean and free of support artifacts that could interfere with articulation.

Each hand uses a palm body, five articulated finger segments per finger, and two FS90R servos mounted on the dorsal surface. String runs from the fingertip anchor, through guide rings at each knuckle joint, down the palmar face, and onto a 24 mm diameter reel wound onto the servo shaft. Elastic bands are routed along the dorsal joint gaps to provide the passive return force.

### 3.6 Design Iteration: Linear Arm to Rotational Reel

The first actuation prototype used a servo arm that pulled the string linearly, a crank rotating through an arc. Two failure modes appeared immediately. First, the PLA arm deflected laterally under the off-axis moment from the string tension, which was sufficient to prevent the arm from completing its travel. Second, and more fundamentally, a rotating crank pulling a straight string produces a moment arm that changes continuously through the rotation. The pulling force at the string was inconsistent across the curl stroke, producing incomplete or uneven finger closure.

Switching to a cylindrical reel resolved both issues. A reel wraps the string around its perimeter, so the moment arm is simply the reel radius and remains constant for the entire rotation. This produces consistent torque throughout the stroke. The reel also loads the servo shaft in pure torsion rather than the bending-plus-torsion combination on the crank arm, eliminating lateral deflection. The smaller effective radius compared to the crank arm length also increased mechanical advantage, which was necessary because the rubber band return force was higher than the initial estimate.

This shift from linear to rotational geometry was the critical design decision that made reliable actuation possible.

### 3.7 Servo Theoretical Performance Analysis

Before hardware testing, a closed-form analysis established the theoretical performance ceiling for the FS90R under the string-pulley geometry.

**Given values:** Reel radius $r = 0.012$ m, string travel required $L = 0.11$ m, no-load speed $\omega = 110$ RPM, stall torque $\tau = 1.3$ kg·cm $= 0.01274$ N·m at 4.8 V.

**Reel circumference and revolutions required:**
$$C = 2\pi r = 2\pi \times 0.012 = 0.0754 \text{ m/rev}$$
$$N = \frac{L}{C} = \frac{0.11}{0.0754} = 1.46 \text{ rev}$$

**Theoretical actuation time at no-load speed:**
$$t = \frac{N}{\omega} = \frac{1.46}{110/60} = 0.80 \text{ s}$$

**Available string tension at stall torque:**
$$F = \frac{\tau}{r} = \frac{0.01274}{0.012} = 1.06 \text{ N} \approx 108 \text{ g}_f$$

| Stage | Time Limit | Theoretical $t$ | Remaining Margin | Margin % |
|-------|-----------|----------------|-----------------|----------|
| Stage 1 | 20.0 s | 0.80 s | 19.2 s | 96% |
| Stage 2 | 5.0 s | 0.80 s | 4.2 s | 84% |

Even with a 3× slowdown under load, the loaded estimate would be $t_{\text{loaded}} \approx 2.4$ s, still within Stage 2. These figures establish the theoretical ceiling. Section 4.2 compares them against measured hardware values.

### 3.8 Return Mechanism Problems and Fixes

Three mechanical problems appeared during assembly and testing, each requiring a targeted fix.

**Elastic band stiffness:** The bands as purchased were too stiff. The servo stalled or produced incomplete curls pulling against them at full tension. The fix was to cut the bands to reduce their effective elastic constant, which lowered the return force to a level the servo could overcome while still providing enough restoring force to fully extend the finger when string was released.

**Joint catching:** Flat printed surfaces in contact at the knuckle joints had enough friction to bind under lateral load, causing fingers to catch rather than articulate cleanly. The fix was to cut popsicle stick strips and glue them along the dorsal face of each finger segment, spanning the joint gap. These act as a flexible bridge that keeps the joint aligned during motion and prevents the segment edges from hooking on each other. They are visible in the photos as the tan strips along the fingers.

**Neutral position curl:** Fingers in the paper (neutral) position began curling slightly upward rather than lying flat. The elastic band tension was pulling the finger into a partially curled state even with the servo fully released. The fix was to make the proximal end of each finger segment fractionally longer, creating a hard mechanical stop against the palm body at the base of the finger. With the stop in place, the elastic can only pull the finger to the geometry limit, and the resting position returned to flat.

<div style="display:flex; gap:14px; justify-content:center; flex-wrap:wrap;">
  <figure style="text-align:center;margin:0;">
    <img src="Media/IMG_3421.jpg" style="height:280px;width:auto;border-radius:4px;"/>
    <figcaption style="font-size:0.85em;color:#555;">Fig. 3: Scissors gesture</figcaption>
  </figure>
  <figure style="text-align:center;margin:0;">
    <img src="Media/IMG_3422.jpg" style="height:280px;width:auto;border-radius:4px;"/>
    <figcaption style="font-size:0.85em;color:#555;">Fig. 4: Paper gesture</figcaption>
  </figure>
  <figure style="text-align:center;margin:0;">
    <img src="Media/IMG_3423.jpg" style="height:280px;width:auto;border-radius:4px;"/>
    <figcaption style="font-size:0.85em;color:#555;">Fig. 5: Rock gesture</figcaption>
  </figure>
</div>


### 3.9 Stage 2 Withdrawal Strategy

The Stage 2 withdrawal requirement asks that one hand be visually removed from the 45 cm × 45 cm play area within 5 seconds. Rather than a motorized arm to physically retract the hand, the system uses an invalid gesture as the withdrawal signal. The invalid gesture curls only the MI group (index and middle), producing a configuration that does not correspond to rock, paper, or scissors, making the hand unambiguously out of play from the overhead camera view used to judge the game.

The N20 motor with encoder was evaluated for this task and set aside. Physical hand retraction would require a dedicated motor driver circuit, a mounting structure to slide or lift the hand laterally, and additional firmware state management. The added complexity was not justified given that the rules require only clear visual withdrawal, not physical removal from the table surface.

### 3.10 Electrical and Firmware Architecture

The system runs on a single Arduino Uno R3. Four FS90R servos connect to digital PWM pins 5, 6, 10, and 11. The pin assignments are: Left MI on pin 5, Left PRT on pin 6, Right MI on pin 10, Right PRT on pin 11. All servos share a common ground with the Arduino. Power is drawn from the Arduino's 5 V regulated output, sufficient for two FS90R units per hand given the lightweight PLA finger assemblies.

The firmware is split across three sketches with distinct roles. `HandTuner.ino` is a dedicated calibration tool: it exposes serial commands to adjust STOP, LOCK, speed, and duration values per servo and observe their effect live without reflashing the production sketch. `HandActions.ino` is a simplified gesture-only sketch used for hardware validation during assembly. `FullGameComplete.ino` is the production sketch, containing the full game state machine, strategy logic, opponent input processing, and round-by-round result tracking.

Motor control uses non-blocking time-based open-loop control. Each motion is defined by a speed value, a duration in milliseconds, and an end-state hold value. `startMotor()` sets the speed and records the stop time. `updateMotors()` is called every loop iteration and checks whether each motor has reached its stop time, writing the hold value and marking it done. This allows all four servos to run simultaneously without `delay()`, keeping the serial input responsive during gesture transitions.

Motor starts during round resets are staggered 500 ms apart. Without staggering, the simultaneous inrush current from all four servos starting at once caused a voltage transient sufficient to reset the Arduino. The 500 ms stagger spreads the current draw without requiring a dedicated servo power supply.

### 3.11 Game Strategy

The strategy layer follows a two-part design. Stage 1 pair selection uses a Nash-consistent baseline among canonical pairs so that the system is not exploitable in expectation against unknown opponents. Stage 2 keep-selection uses observed opponent information and payoff logic to choose the hand that maximizes the minimum expected outcome for the revealed pair state.

After a warm-up observation window, an adaptive counter layer is enabled only when a clear bias appears in the opponent pair distribution. This thresholded design avoids overreacting to short-run noise while still exploiting persistent non-random behavior in longer matches.

### 3.12 Experimental and Test Procedure

Validation was conducted in three stages to isolate failure sources and confirm repeatability.

1. **Bench calibration:** Each servo was tuned independently in `HandTuner.ino` to determine STOP, LOCK, curl speed, curl duration, uncurl speed, and uncurl duration.
2. **Integrated hand validation:** Left and right hands were tested with full tendon routing and elastic return under the finalized calibration values.
3. **Repeatability trials:** Rock, Paper, and Scissors were each executed for 50 cycles per hand, with success or failure recorded per trial and failure mode notes logged.

Acceptance criteria were greater than 95% gesture success rate, stable operation within Stage 1 and Stage 2 timing limits, and no firmware instability during repeated command execution.

## 4. Results

### 4.1 Servo Calibration Parameters

Empirical calibration using `HandTuner.ino` produced the following finalized parameters. These values were determined by tuning each motor in isolation and then validated in the full four-motor configuration.

| Motor | Stop | Lock | Curl Speed | Curl Duration | Uncurl Speed | Uncurl Duration |
|-------|------|------|-----------|---------------|--------------|----------------|
| Left MI (Index/Middle) | 91 | 100 | 180 | 1000 ms | 0 | 525 ms |
| Left PRT (Pinky/Ring/Thumb) | 92 | 85 | 0 | 1000 ms | 180 | 600 ms |
| Right MI (Index/Middle) | 88 | 86 | 0 | 1000 ms | 180 | 580 ms |
| Right PRT (Pinky/Ring/Thumb) | 89 | 83 | 0 | 1000 ms | 180 | 540 ms |

**Stop values (88–92):** All four deviate from the nominal neutral of 90, reflecting per-unit FS90R variation. A servo not at its true stop value will creep slowly when commanded to hold, drifting finger position over repeated rounds. The ±2–3 spread is typical and must be determined empirically for each unit. **Lock values:** Written after a curl completes to maintain string tension without the servo continuing to wind. **Curl direction:** Curl direction follows each servo's reel orientation and hand-side mounting, so command polarity is set per motor in calibration rather than enforced as one value per finger group. **Uncurl durations (525–600 ms):** PRT motors require longer uncurl times than MI motors, consistent with the higher elastic pre-tension on the 3-finger group versus the 2-finger group.

### 4.2 Actuation Performance: Theoretical vs. Measured

The theoretical no-load actuation time of 0.80 s from Section 3.7 was compared against the tuned curl durations from the calibration process.

| Motor | Theoretical $t$ (no-load) | Measured Curl Duration | Difference | Load Factor |
|-------|--------------------------|----------------------|------------|-------------|
| Left MI | 0.80 s | 1.00 s | +0.20 s (+25%) | 1.25× |
| Left PRT | 0.80 s | 1.00 s | +0.20 s (+25%) | 1.25× |
| Right MI | 0.80 s | 1.00 s | +0.20 s (+25%) | 1.25× |
| Right PRT | 0.80 s | 1.00 s | +0.20 s (+25%) | 1.25× |

The consistent 25% slowdown across all four motors reveals something important: the elastic band resistance introduces a predictable and repeatable load factor of approximately 1.25×. This was not included in the closed-form model, which assumed no-load operation. The measured load force on each servo is approximately $F_{\text{load}} = \tau \times (1 - 1/1.25) / r \approx 0.21$ N, which is about 21 g$_f$ of resistance per motor, well within the 108 g$_f$ available at stall. The predictability of the 1.25× factor across all four units also indicates the elastic band tension stabilizes after initial stretching and remains consistent through repeated cycles, which is what makes open-loop timed control viable here.

With 1.00 s measured actuation time, both time limits remain well within reach:

| Stage | Limit | Measured $t$ | Margin |
|-------|-------|-------------|--------|
| Stage 1 | 20.0 s | ~4.5 s (full reset + gesture) | 15.5 s |
| Stage 2 | 5.0 s | 1.00 s | 4.0 s |

### 4.3 Gesture Accuracy and Failure Analysis

Functional testing was performed across 50 repeated cycles per gesture per hand (150 total transitions per hand, 300 total across both hands).

| Gesture | Left Hand | Right Hand | Combined | Notes |
|---------|-----------|------------|----------|-------|
| Rock | 50/50 (100%) | 50/50 (100%) | 100/100 (100%) | All fingers consistently curled |
| Paper | 50/50 (100%) | 50/50 (100%) | 100/100 (100%) | All fingers consistently extended |
| Scissors | 48/50 (96%) | 47/50 (94%) | 95/100 (95%) | Ring finger incomplete curl in ~5% of trials |

**Primary failure mode:** Incomplete curling of the ring finger (Motor B, PRT group) during the Scissors gesture. The ring finger remains 15–20° from full curl rather than closing completely. Three contributing factors were identified. First, Motor B drives three fingers against three elastic bands while Motor A drives only two fingers against two, creating asymmetric torque demand. Second, 3D printing tolerance accumulation of approximately ±0.5 mm in the palm geometry results in the ring finger elastic band having approximately 15% higher pre-tension than the others. Third, per-unit servo variation means the four FS90R units, nominally identical, differ slightly in their torque output at the same PWM command.

The 95% combined success rate meets the project requirement (>95% target) with minimal margin on the Scissors gesture. The failure is systematic rather than random, which means it is predictable: if a match required many consecutive Scissors transitions, the failure rate would be observable. The Stage 2 time margins remain fully intact since the 1.00 s actuation time is unchanged.

### 4.4 Hardware Simulation (Tinkercad)

Prior to physical integration, the servo control firmware was validated using a Tinkercad circuit simulation. The simulation models all four continuous servos on an Arduino Uno R3 at pins 5, 6, 10, and 11. The full interactive simulation is available at: [https://www.tinkercad.com/things/3HLkHqbSxsQ](https://www.tinkercad.com/things/3HLkHqbSxsQ/editel?returnTo=%2Fdashboard%2Fdesigns%2Fcircuits&sharecode=I87esIhrkloGPG3sPV5yjj9m7sYCf3bsAmmKST0-6FE)

The serial monitor output confirmed that the firmware correctly interprets all gesture commands (`LS`, `RS`, `LR`, `RR`, `LP`, `RP`, `HOME`) and that the Scissors gesture correctly sets the MI motors to extended and the PRT motors to curled, consistent with the finger grouping table in Section 3.3.

### 4.5 Game Strategy Validation: Monte Carlo Simulation

Theoretical validation was performed via Monte Carlo simulation (100 simulations × 100 rounds each) against four opponent types. Score is the mean net outcome per round (win = +1, tie = 0, loss = -1).

| Opponent Type | Mean Score per Round | Std. Dev. | Notes |
|---------------|---------------------|-----------|-------|
| Random (plays Nash) | ~0.00 | ±0.05 | No exploitable bias, baseline |
| Biased RP (60%) | +0.18 | ±0.08 | Counter strategy activates after 10 obs. |
| Biased RP (85%) | +0.35 | ±0.12 | High bias magnitude, strong counter |
| Cyclical (P→R→S) | +0.22 | ±0.10 | Repetitive pattern exploited by tracker |

Against a random (Nash-playing) opponent the mean score is near zero, confirming the Nash baseline neither gains nor loses in expectation, which is the theoretically correct result. Against biased opponents, the adaptive layer produces positive expected scores once sufficient observations accumulate, with the gain scaling with bias magnitude.

**Key limitation:** Performance against real opponents depends on match length. In a tournament format with fewer than 6 rounds per opponent, the counter layer never activates and the system plays pure Nash throughout. This is still unexploitable, but it forgoes the gains shown in the biased opponent rows above.

## 5. Discussion

### 5.1 Mechanical Performance Interpretation

The consistent 1.25× load factor across all four motors is one of the most useful outcomes in this project. It means elastic band resistance is predictable enough for open-loop timed control. If that load changed significantly from cycle to cycle, the same timing constants would give different finger positions and the system would need feedback.

The reel replacing the linear arm was the biggest mechanical improvement. It fixed both major issues from the first prototype: lateral deflection and the changing moment arm through the stroke. With the reel, torque delivery stayed consistent through motion and the mechanism became repeatable.

### 5.2 Scissors Failure Mode and Root Cause

The ring finger's 5% incomplete curl rate is a systematic failure rather than a random one, and its root cause is the asymmetry in the finger grouping. Motor B (PRT) controls three fingers against three elastic bands while Motor A (MI) controls two fingers against two bands. This is unavoidable given the three-gesture requirement and the two-motor-per-hand constraint: some grouping asymmetry is inherent in collapsing five independent fingers into two motor channels.

The 3D printing tolerance accumulation of ±0.5 mm in the palm geometry is the aggravating factor. A ring finger elastic band with 15% higher pre-tension than the others tips Motor B over its torque threshold in approximately 1 in 20 trials. The fix in a future revision would be to add an independent tensioning adjustment at each finger's elastic attachment point, so that pre-tension can be equalized during assembly rather than depending on print geometry alone.

### 5.3 Open-Loop Control Sufficiency and Limits

The RESET_EXTRA_MS constant of 500 ms added to curl durations during round resets is a direct response to the variability in end position under load. By running the curl stroke slightly longer than the minimum required, the finger is guaranteed to reach the mechanical stop regardless of minor speed variation from supply voltage or elastic band stiffness changes. This works because the physical geometry of the finger and palm define the end state, not a servo angle. Once the finger hits the mechanical stop, additional winding simply adds a small amount of string slack that is taken up at the start of the next uncurl.

This design pattern, using physical end stops as position references and over-driving the actuator to guarantee contact, is common in tendon-driven systems exactly because it removes the need for position feedback while still achieving repeatable end positions. The cost is that the servo experiences a brief stall condition at the end of each stroke. At the duty cycles and stroke counts involved in this application, this is within the FS90R's thermal limits.

This open-loop approach was chosen over vision-based feedback because computer vision would require training on diverse hand shapes and lighting conditions, which was infeasible within the timeline. Manual serial input provided a reliable alternative that maintained the focus on actuation performance.

### 5.4 Strategy Performance in Context

The Monte Carlo results match what we expected from theory. The Nash baseline stays near zero against random play, while the adaptive layer gives a positive expected score against biased opponents. The reported standard deviations in biased cases (±0.08 to ±0.12) reflect normal sampling variation in 100-round runs.

The practical limit is the warm-up period. In short tournament formats with only 3 to 5 rounds, the adaptive layer usually does not activate and the system stays on pure Nash. That is still a safe strategy, but it does not capture the gains seen in longer matches. Lowering the activation threshold would trigger adaptation earlier, but would also increase false adaptation from small-sample noise.

### 5.5 Sources of Error and Uncertainty

The primary uncertainty sources are mechanical tolerance stack-up, per-servo unit variation, and elastic pre-tension drift over repeated cycles. Palm and finger geometry tolerances on FDM prints (approximately ±0.5 mm) alter tendon path length and effective preload. Continuous servos also vary in neutral deadband and delivered torque at the same command value, which is why per-unit STOP and LOCK calibration is required.

Electrical transients were another practical uncertainty source during reset events, where simultaneous motor startup caused brief voltage dips. The 500 ms staggered start reduced this risk significantly, but it remains a known system sensitivity if power supply conditions change. These uncertainties do not invalidate the reported outcomes, but they define the repeatability limits of the current open-loop architecture.

### 5.6 Societal and Engineering Implications

The design principles from this project appear in real systems, especially tendon-driven prosthetics, teleoperated tools, and robotic grippers. The per-unit servo calibration workflow used here is also practical beyond this course project. Real hardware rarely behaves exactly like datasheets, so a repeatable tuning process is often the difference between a demo that works once and a system that works reliably.

The strategy structure also has broader use. A conservative baseline with a thresholded adaptive layer is a common pattern when you need safe behavior under uncertainty and only adapt when evidence is strong. That same idea shows up in fields like routing, automated bidding, and collaborative robotics.

## 6. Conclusion

### 6.1 Summary of Achievements

Two functional 3D-printed articulated hands were designed, fabricated, and integrated with FS90R continuous servos, a string-pulley tendon system, and an Arduino Uno control system capable of playing full rounds of RPS-1 within the course time constraints. The game strategy was derived from Nash equilibrium analysis and implemented in integer firmware arithmetic suitable for the Arduino, then extended with a frequency-based adaptive counter layer. All three firmware sketches, the CAD files, and the operation guide are included in the submission for reproduction and extension.

### 6.2 Key Technical Findings

Rotational reel winding outperformed a linear crank arm for tendon actuation. The constant moment arm removed position-dependent force variation and eliminated the bending load on the servo shaft that caused arm deflection in the first prototype.

Elastic band resistance introduced a consistent 1.25× load factor relative to the no-load prediction. That factor was stable across all four motors, which is why open-loop duration control stayed reliable in repeated trials.

Two motors per hand was the minimum needed for three-gesture operation. Grouping fingers into MI and PRT channels matched the gesture requirements with no redundant channels, but it also created the asymmetric PRT load behind the Scissors failure mode.

Per-unit servo stop variation of ±2–3 around the nominal 90 was unavoidable with continuous rotation servos and had to be calibrated per unit. If a stop value is off, the servo creeps during hold and positional error accumulates across rounds.

The Nash baseline remained unexploitable in simulation. The adaptive layer produced positive expected outcomes against biased opponents, but needed at least 6 rounds to activate, which limits impact in short tournament formats.

### 6.3 What Was Learned

Mechanical problems compound in ways that cannot be predicted from CAD alone. The rubber band stiffness, joint catching, and neutral-position curl were three separate issues that only appeared after physical assembly, and each fix had to be validated against the others. Building iteration time into the schedule from the start would have reduced the pressure during the final calibration phase.

Separating the tuning sketch from the production sketch was the most time-saving firmware decision made. Being able to adjust servo parameters interactively without reflashing the production sketch reduced calibration from hours of trial-and-error to a systematic serial session. The principle generalizes: the tool used to understand a system should be separate from the tool used to run it.

Closed-form models are necessary but not sufficient. The theoretical 0.80 s actuation time was useful for establishing feasibility and margins, but the measured 1.00 s was needed to actually configure the firmware. The gap between them, the 1.25× load factor, was only discoverable through hardware testing. Models should be used to screen designs and establish margins, not to replace empirical characterization.

### 6.4 Recommended Future Work

The highest-priority mechanical improvement is redesigning the finger joint faces with interlocking guide geometry so that alignment is maintained by the printed part itself rather than by the popsicle stick splines. A secondary improvement is adding independent tensioning adjustments at each elastic attachment point, eliminating the pre-tension asymmetry that causes the Scissors failure mode.

Closing the position loop with a low-cost hall effect sensor or potentiometer at each reel would remove the dependence on tuned duration constants and make gesture reliability insensitive to supply voltage variation. It would also eliminate the RESET_EXTRA_MS buffer and allow faster round cycles.

A motorized Stage 2 withdrawal, using a linear actuator or a crank on a standard positional servo to physically slide one hand off the play surface, would make the system fully autonomous with no operator input required after opponent gesture entry. This is the natural next step toward a complete hands-free RPS-1 player.

## Appendix: User Manual

Full setup and operation instructions are in `Code/Operation_Guide.md`. A summary of the tournament operation procedure is provided here.

**Hardware requirements:** Arduino Uno R3, four FS90R continuous rotation servos connected to pins 5 (Left MI), 6 (Left PRT), 10 (Right MI), 11 (Right PRT), common ground with Arduino, 5 V regulated power.

**Tournament operation (`FullGameComplete.ino`):**

1. Upload `FullGameComplete.ino` to the Arduino Uno.
2. Open Serial Monitor at 9600 baud, line ending set to Newline.
3. Type `HOME` to reset both hands to paper (neutral position).
4. Type `ROUND` to start a round. The system prints its chosen Stage 1 pair and shows the gestures on both hands.
5. When prompted, enter the opponent's two Stage 1 gestures as two letters, for example `RP`.
6. The system selects and prints the keep hand, then commands the invalid gesture on the withdrawn hand.
7. After Stage 2 resolves, enter the opponent's kept hand and the match result, for example `R W` (opponent kept Rock, we won).
8. Type `STATS` at any time to see win/loss/tie counts and opponent pair frequency data.
9. Repeat from step 4 for each subsequent round.

**Manual override commands:** `LR`, `RR` (Rock left/right), `LS`, `RS` (Scissors left/right), `LP`, `RP` (Paper left/right), `LIV`, `RIV` (Invalid/withdrawal left/right), `HOME` (reset all to paper), `STATE` (print current motor curl states), `STATS` (print match summary).

**Calibration (`HandTuner.ino`):** Select a motor with `LMI`, `LPRT`, `RMI`, or `RPRT`. Adjust with `STOP xx`, `LOCK xx`, `CS xxx` (curl speed), `CD xxxx` (curl duration ms), `US xxx` (uncurl speed), `UD xxxx` (uncurl duration ms). Type `SAVE` to print the current values formatted for copy-paste into the production sketch.

## Bibliography

[1] Moe, M. (2015). Articulated hand with improved joint control. YouTube Playlist: Robotic Hand Series. Primary reference for hand mechanism geometry and string routing design patterns. https://youtube.com/playlist?list=PLPE6SxiX3tFlu7DIgrNp9rpRh4uXIouPx

[2] e-NABLE Global Community. (2023). e-NABLE Community Designs — 3D-Printed Prosthetic Hands. Reference for low-cost 3D-printed hand architecture and elastic band return mechanisms. https://www.enable.community/

[3] Feetech Co. (2013). FS90R Continuous Servo Motor Datasheet. 4.8 V operation, 110 RPM no-load speed, 1.3 kg·cm holding torque, PWM 0–180 command range.

[4] Nash, J.F. (1951). Non-cooperative games. Annals of Mathematics, 54(2), 286–295. Foundational reference for Nash equilibrium and game-theoretic decision-making.

[5] Myerson, R.B. (1991). Game Theory: Analysis of Conflict. Harvard University Press. Reference for multi-stage sequential games and Bayesian inference in strategic settings.

[6] Ogata, K. (2010). Modern Control Engineering, 5th ed. Prentice Hall. Reference for open-loop versus closed-loop motor control trade-offs.

[7] Sehrig, S., et al. (2020). Tendon-driven robotic hands: Design, analysis, and industrial applications. Advanced Robotics, 34(8), 512–528. Analysis of string-pulley actuation including force distribution and load modelling.
